# Tutorial: Milestone 5 - LLM Skill Orchestrator (Phase 1)

This notebook validates the new `skills/lucida-orchestrator` package against live daemon behavior.

Expected outcomes:
- CLI flow succeeds (`session.create -> dataset.open -> view.create -> render.image`).
- HTTP flow succeeds with equivalent assertions.
- Failure-path checks return expected error codes (`view_not_found`, `unsupported_mode`, `render_output_too_large`).


## Prerequisites

- Run from inside this repository.
- `uv sync --dev` has completed.
- Rust toolchain is available for daemon startup when needed.


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

import httpx


In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
BASE_URL = os.environ.get("LUCIDA_BASE_URL", "http://127.0.0.1:3000")
TMP_DIR = REPO_ROOT / "tmp" / "notebooks" / "milestone5-skill"
TMP_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT / "tests" / "python"))
from parity.data_setup import create_sample_omezarr


def wait_for_health(base_url: str, timeout_s: float = 30.0) -> bool:
    deadline = time.monotonic() + timeout_s
    with httpx.Client(timeout=2.0) as client:
        while time.monotonic() < deadline:
            try:
                response = client.get(f"{base_url}/healthz")
                if response.status_code == 200 and response.json().get("status") == "ok":
                    return True
            except Exception:
                pass
            time.sleep(0.5)
    return False


def ensure_daemon(base_url: str) -> subprocess.Popen[str] | None:
    if wait_for_health(base_url, timeout_s=2.0):
        return None

    process = subprocess.Popen(
        ["cargo", "run", "-p", "lucida-daemon"],
        cwd=REPO_ROOT,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        text=True,
    )
    if not wait_for_health(base_url, timeout_s=45.0):
        process.terminate()
        raise RuntimeError("Failed to start lucida-daemon")
    return process


def run_cli_json(command: list[str]) -> dict[str, Any]:
    env = os.environ.copy()
    env["LUCIDA_BASE_URL"] = BASE_URL
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
        env=env,
    )
    return json.loads(completed.stdout)


In [3]:
_daemon_process = ensure_daemon(BASE_URL)
DATASET_URI = create_sample_omezarr(str(TMP_DIR / "phase1-skill-smoke.zarr"))
print({"repo_root": str(REPO_ROOT), "base_url": BASE_URL, "dataset_uri": DATASET_URI, "daemon_started_here": _daemon_process is not None})


{'repo_root': '/Users/austin/GitHub/lucida', 'base_url': 'http://127.0.0.1:3000', 'dataset_uri': '/Users/austin/GitHub/lucida/tmp/notebooks/milestone5-skill/phase1-skill-smoke.zarr', 'daemon_started_here': False}


## Step 1 - CLI workflow

Run a full phase-1 path using `uv run lucida ... --json` and assert key fields.


In [4]:
cli_session = run_cli_json(["uv", "run", "lucida", "session", "create", "--json"])
session_id = str(cli_session["session_id"])
assert session_id

cli_dataset = run_cli_json([
    "uv", "run", "lucida", "dataset", "open",
    "--uri", DATASET_URI,
    "--dataset-id", "nb_skill_dataset",
    "--session-id", session_id,
    "--json",
])
dataset_id = str(cli_dataset["dataset_summary"]["dataset_id"])
assert dataset_id == "nb_skill_dataset"

cli_view = run_cli_json([
    "uv", "run", "lucida", "view", "create",
    "--dataset-id", dataset_id,
    "--session-id", session_id,
    "--mode", "2d",
    "--json",
])
view_id = str(cli_view["view_state"]["view_id"])
assert view_id

cli_render = run_cli_json([
    "uv", "run", "lucida", "render", "image",
    "--view-id", view_id,
    "--session-id", session_id,
    "--width-px", "192",
    "--height-px", "128",
    "--delivery", "inline_base64",
    "--json",
])
assert cli_render["status"] == "ok"
assert cli_render["images"][0]["sha256"]
print({"session_id": session_id, "dataset_id": dataset_id, "view_id": view_id, "render_status": cli_render["status"]})


{'session_id': 'session_a2e3da946253449f', 'dataset_id': 'nb_skill_dataset', 'view_id': 'view_e9f8972cc6234092', 'render_status': 'ok'}


## Step 2 - HTTP workflow

Run equivalent flow against daemon routes and validate response fields.


In [5]:
with httpx.Client(base_url=BASE_URL, timeout=20.0) as client:
    http_session = client.post("/session/create", json={"schema_version": 1})
    http_session.raise_for_status()
    http_session_id = str(http_session.json()["session_id"])

    http_dataset = client.post(
        "/dataset/open",
        json={
            "schema_version": 1,
            "uri": DATASET_URI,
            "dataset_id": "nb_skill_dataset_http",
            "session_id": http_session_id,
        },
    )
    http_dataset.raise_for_status()
    http_dataset_id = str(http_dataset.json()["dataset_summary"]["dataset_id"])

    http_view = client.post(
        "/view/create",
        json={
            "schema_version": 1,
            "dataset_id": http_dataset_id,
            "session_id": http_session_id,
            "mode": "2d",
        },
    )
    http_view.raise_for_status()
    http_view_id = str(http_view.json()["view_state"]["view_id"])

    http_render = client.post(
        "/render/image",
        json={
            "schema_version": 1,
            "view_id": http_view_id,
            "session_id": http_session_id,
            "output": {
                "format": "png",
                "delivery": "inline_base64",
                "width_px": 192,
                "height_px": 128,
            },
        },
    )
    http_render.raise_for_status()
    render_payload = http_render.json()

assert render_payload["status"] == "ok"
assert render_payload["images"][0]["sha256"]
print({"http_session_id": http_session_id, "http_dataset_id": http_dataset_id, "http_view_id": http_view_id})


{'http_session_id': 'session_c4745a0592c74d6b', 'http_dataset_id': 'nb_skill_dataset_http', 'http_view_id': 'view_8b3001a102a4421c'}


## Step 3 - Failure-path checks

Verify troubleshooting codes documented in `skills/lucida-orchestrator/references/troubleshooting.md`.


In [6]:
with httpx.Client(base_url=BASE_URL, timeout=20.0) as client:
    missing_view = client.post(
        "/view/update",
        json={
            "schema_version": 1,
            "view_id": "view_missing",
            "session_id": http_session_id,
            "patch": [{"op": "replace", "path": "/view_2d/camera/zoom", "value": 1.1}],
        },
    )
    assert missing_view.status_code == 404
    assert missing_view.json()["code"] == "view_not_found"

    unsupported_mode = client.post(
        "/view/create",
        json={
            "schema_version": 1,
            "dataset_id": http_dataset_id,
            "session_id": http_session_id,
            "mode": "3d",
        },
    )
    assert unsupported_mode.status_code == 422
    assert unsupported_mode.json()["code"] == "unsupported_mode"

    oversized_render = client.post(
        "/render/image",
        json={
            "schema_version": 1,
            "view_id": http_view_id,
            "session_id": http_session_id,
            "output": {
                "format": "png",
                "delivery": "inline_base64",
                "width_px": 5000,
                "height_px": 5000,
            },
        },
    )
    assert oversized_render.status_code == 422
    assert oversized_render.json()["code"] == "render_output_too_large"

print("failure-path assertions passed")


failure-path assertions passed


In [7]:
if _daemon_process is not None:
    _daemon_process.terminate()
    _daemon_process.wait(timeout=10)
    print("stopped notebook-managed daemon")
else:
    print("daemon was pre-existing; left running")


daemon was pre-existing; left running
